# **Automatic Alternate Abbreviation Annotation:** LLM predictions on gene-alias pairs for Alternate Abbreviation alias symbols

This notebook runs the Alternate Abbreviation annotation workflow on a dataset of gene alias symbols. Alias symbols are first filtered using heuristic rules, after which those requiring further review are submitted to the LLM for annotation. Results are cached and stored for downstream analysis. For more information on Alternate Abbreviations and the overall workflow, see [README](alternate_abbreviation_README).

In [1]:
import re
from pathlib import Path
import pickle
import polars as pl
from typing import Any
from tqdm.notebook import tqdm
from wags_llm.cache import InMemoryCache
from wags_llm.client import BedrockClaudeJsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner
from rapidfuzz.distance import LCSseq
from alt_abbrev_models import AlternateAbbreviationPredictionResult
from alt_abbrev_models import MatchedRule
from alt_abbrev_models import RuleResult
from alt_abbrev_models import AlternateAbbreviationPrompt

In [2]:
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
REGION_NAME = "us-east-1"
PROFILE_NAME = "dev-account"
MAX_TOKENS = 150


def build_llm_task_runner(
    model_id: str,
    region_name: str,
    profile_name: str,
    max_tokens: int,
    temperature: float,
) -> StructuredTaskRunner:
    """Build LLM alternate abbreviation alias annotator

    :param model_id: Bedrock model identifier.
    :param region_name: AWS region for the Bedrock runtime client.
    :param profile_name: AWS profile name.
    :param max_tokens: Maximum number of tokens to request from the model.
    :param temperature: Sampling temperature.
    :return: Configured structured task runner instance.
    """
    registry = build_empty_registry()
    registry.register(AlternateAbbreviationPrompt(version="v1"))
    llm_client = BedrockClaudeJsonClient(
        model_id=model_id,
        region_name=region_name,
        profile_name=profile_name,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    cache = InMemoryCache()
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=cache
    )

In [3]:
def rule_based_evaluation(
        df: pl.DataFrame,
        alias_symbol: str, 
        primary_gene_symbol: str, 
        gene_name: str, 
        threshold: float = 0.20,
    ) -> RuleResult:
    """Evaluate the alias gene symbol using rules. If the alias meets the criteria, it will be marked as NOT an alternate abbreviation and will NOT be sent to the LLM for evaluation. 

    Applies a series of heuristic filters to the alias symbol
    - checks for captured categories "Gene Identifier Symbol" and "Clone Symbol", not an alternate abbreviation alias
    because it is an identifier being used as an alias
    - 3 or more extra characters in the alias, not in the gene name
    - longest common subsequence (LCS) similarity to the primary gene symbol to be at
    least whatever the threshold is set to
    
    :param df: DataFrame containing alias annotation metadata
    :param alias_symbol: The alias gene symbol being evaluated
    :param primary_gene_symbol: The official primary gene symbol
    :param gene_name: The full gene name associated with the gene
    :param threshold: Minimum LCS similarity score required to pass filtering
    :return: RuleResult containing:
        - lcs_similarity_score: float similarity score between alias and primary gene symbol
        - num_extra_characters: number of extra characters in alias not present in gene name
        - conflicting_category: list of matched captured categories if rule is triggered, otherwise None
        - matched_rule: MatchedRule if a rule is triggered, otherwise None
    """

    def calc_lcs_similarity(s1: str, s2: str) -> float:
        """Calculate the longest common subsequence (LCS) similarity

        :param s1: the first string, alias symbol
        :param s2: the second string, primary gene symbol
        :return: value between 0 and 1
        """
        return LCSseq.normalized_similarity(s1, s2)

    def count_extra_characters(alias: str, gene_name: str) -> int:
        """Denote if the alias symbol has characters not present in the gene name

        :param alias: gene alias symbol
        :param gene_name: HGNC designated official gene name
        :return: Number of extra characters
        """
        norm_alias = re.sub(r"[^A-Z0-9]", "", alias.upper())

        alias_set = set(norm_alias)
        name_set = set(gene_name)

        extra_chars = alias_set - name_set
        return len(extra_chars)
    
    def find_conflicting_category(
        df: pl.DataFrame,
        alias: str,
        primary: str,
    ):
        """Return conflicting category if alias row matches rule, else None.
        
        :param df: input dataframe containing alias annotations
        :param alias: alias gene symbol (normalized)
        :param primary: primary gene symbol (normalized)
        :return: list of conflicting categories if present, otherwise None
        """

        normalized = df.with_columns(
            pl.col("captured_category_list")
            .cast(pl.Utf8)
            .str.replace_all(r"[\[\]']", "")
            .str.split(r",\s*")
            .alias("captured_category_list_norm")
        )

        result = normalized.filter(
            (pl.col("gene_symbol").str.to_uppercase() == alias) &
            (pl.col("primary_gene_symbol").str.to_uppercase() == primary) &
            (
                pl.col("captured_category_list_norm").list.contains("Gene Identifier Symbol") |
                pl.col("captured_category_list_norm").list.contains("Clone Symbol")
            )
        )

        if result.is_empty():
            return None

        return (
            result
            .select("captured_category_list_norm")
            .explode("captured_category_list_norm")
            .unique()
            .get_column("captured_category_list_norm")
            .to_list()
        )

    alias = alias_symbol.upper()
    primary = primary_gene_symbol.upper()
    name = gene_name.upper()

    extra_count = count_extra_characters(alias, name)

    lcs_score = calc_lcs_similarity(alias, primary)

    conflicting_category = find_conflicting_category(df, alias, primary)

    if conflicting_category:
        return RuleResult(
            conflicting_category=conflicting_category,
            matched_rule=MatchedRule.CONFLICT_CATEGORY
        )
    
    # 3 or more extra characters in the alias that is not present in the gene name indicates that the alias is referring to something other than what the gene name is representing.
    # 1 or 2 extra characters may be acceptable since there are cases where a word is implied but not present in the gene name 
    # Ex: the gene TRBV11-3, T cell receptor beta variable 11-3, has an alias (TCRBV11S3) that IS an alternate abbreviation where S represents "segment"
    # The gene is the third segment in the 11th gene family
    # Another example could be that the name refers to is as ABC-A but the alternate abbreviation is ABC-1 where the difference is using an alphabetical character vs a numerical character to indicate the gene as the first
    if extra_count >= 3:
        return RuleResult(
            lcs_similarity_score=None,
            num_extra_characters=extra_count,
            conflicting_category=None,
            matched_rule=MatchedRule.EXTRA_CHARACTERS
        )

    if lcs_score < threshold:
        return RuleResult(
            lcs_similarity_score=lcs_score,
            num_extra_characters=None,
            conflicting_category=None,
            matched_rule=MatchedRule.LOW_LCS_SIMILARITY
        )

    return RuleResult(
        lcs_similarity_score=lcs_score,
        num_extra_characters=extra_count,
        conflicting_category=None,
        matched_rule=None
    )

In [4]:
def get_alt_abbreviation_annotation(
    df: pl.DataFrame,
    task_runner: StructuredTaskRunner,
    prompt: AlternateAbbreviationPrompt,
    gene_symbol: str,
    primary_gene_symbol: str,
    gene_name: str,
    hgnc_id: str,
) -> AlternateAbbreviationPredictionResult:
    """Determine whether a gene symbol is an alternate abbreviation of a
    primary HGNC gene symbol.

    Alias symbols are first evaluated using rule-based criteria. If a rule
    determines that the alias is not an alternate abbreviation, a result
    containing the matched rule and similarity score is returned. Otherwise,
    the configured prompt is executed and the resulting annotation is returned
    along with the computed similarity score.
    
    :param df: input dataframe containing alias annotations
    :param task_runner: Structured task runner used to execute the LLM prompt.
    :param prompt: Configured prompt instance for alternate abbreviation annotation.
    :param gene_symbol: Candidate alias symbol to evaluate.
    :param primary_gene_symbol: Primary HGNC-approved gene symbol being
        compared against.
    :param gene_name: Full gene name associated with the primary gene symbol.
    :param hgnc_id: HGNC identifier for the primary gene.
    :return: Annotation result containing the rule-based decision or LLM
        prediction, similarity score, and any error information.
    """

    rule_result = rule_based_evaluation(
        df,
        gene_symbol,
        primary_gene_symbol,
        gene_name,
    )

    if rule_result.matched_rule is not None:
        return AlternateAbbreviationPredictionResult(
            matched_rule=rule_result.matched_rule,
            lcs_similarity_score=rule_result.lcs_similarity_score,
            num_extra_characters=rule_result.num_extra_characters,
            conflicting_category=rule_result.conflicting_category,
            llm_annotation=None,
            error_message=None,
        )

    payload = prompt.build_payload(
        gene_symbol=gene_symbol,
        primary_gene_symbol=primary_gene_symbol,
        gene_name=gene_name,
        hgnc_id=hgnc_id,
    )

    try:
        task_result = task_runner.execute(
            prompt_name=prompt.name,
            prompt_version=prompt.version,
            payload=payload,
            response_model=AlternateAbbreviationPredictionResult,
        )

        return AlternateAbbreviationPredictionResult(
            llm_annotation=task_result.llm_annotation,
            lcs_similarity_score=rule_result.lcs_similarity_score,
        )

    except Exception as e:
        return AlternateAbbreviationPredictionResult(
            error_message=str(e),
            lcs_similarity_score=rule_result.lcs_similarity_score,
        )

In [5]:
def run_experiments(
    df: pl.DataFrame,
    temperatures: list[float],
    num_runs: int,
    prompt_version: str,
) -> list[dict[str, Any]]:
    """Run LLM annotation experiments across a range of temperatures and multiple runs per temperature, storing results for analysis.

    :param df: Input dataframe containing gene symbols and associated information.
    :param temperatures: List of temperature values to experiment with.
    :param num_runs: Number of runs to execute for each temperature setting.
    :param prompt_version: Version of the prompt template to use for annotation.
    :return: List of stored runs with LLM outputs and diagnostics for each temperature and run
    """
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                MODEL_ID,
                REGION_NAME,
                PROFILE_NAME,
                MAX_TOKENS,
                temp,
            )

            prompt = AlternateAbbreviationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for row in tqdm(
                df.iter_rows(named=True),
                total=df.height,
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                result = get_alt_abbreviation_annotation(
                    df,
                    task_runner=task_runner,
                    prompt=prompt,
                    gene_symbol=row["gene_symbol"],
                    primary_gene_symbol=row["primary_gene_symbol"],
                    gene_name=row["gene_name"],
                    hgnc_id=row["HGNC_ID"],
                )

                if result.error_message:
                    raise RuntimeError(
                        f"""
                        Error in temp={temp}, run={run_idx + 1}
                        gene_symbol={row["gene_symbol"]}
                        primary_gene_symbol={row["primary_gene_symbol"]}
                        HGNC_ID={row["HGNC_ID"]}
                        error={result.error_message}
                        """
                    )

                results.append(
                    {
                        "llm_annotation": result.llm_annotation,
                        "matched_rule": result.matched_rule,
                        "error_message": result.error_message,
                        "lcs_similarity_score": result.lcs_similarity_score,
                        "gt": row["alternate_abbreviation_status"],
                    }
                )
            print("prompt_version being stored:", prompt_version)

            stored_runs.append(
                {
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "temperature": temp,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [6]:
def build_experiment_key(sample_name, prompt_version, temperatures, num_runs):
    """Build a unique key for identifying an experiment configuration based on sample name, prompt version, temperatures, and number of runs.

    :param sample_name: Name of the sample or dataset being used.
    :param prompt_version: Version of the prompt template used in the experiment.
    :param temperatures: List of temperature values used in the experiment.
    :param num_runs: Number of runs executed for each temperature setting.
    :return: A string key uniquely identifying the experiment configuration.
    """
    temp_str = "-".join(str(t).replace(".", "p") for t in temperatures)
    return f"{sample_name}_p{prompt_version}_t{temp_str}_agg{num_runs}"

In [7]:
ALT_ABBREV_ROOT = Path.cwd().resolve()
ALT_ABBREV_OUTPUT_PATH = ALT_ABBREV_ROOT / "output"

## A test is recommended before using resources to run full dataset

In [8]:
RUN_SUBSET = False

### Load dataset with gene-alias pairs manually curated for Alternate Abbreviation alias symbols

In [9]:
df = pl.read_excel(
    ALT_ABBREV_OUTPUT_PATH / "alt_abbrev_annotation_manually_annotated_df.xlsx"
)

### Create a truncated version of the dataset to test

In [10]:
if RUN_SUBSET:
    test_df = df.head(30)
    SAMPLE_PATH = Path(
        ALT_ABBREV_OUTPUT_PATH
        / "subset_alt_abbrev_annotation_manually_annotated_df.xlsx"
    )
    test_df.write_excel(SAMPLE_PATH)
    df = test_df
else:
    SAMPLE_PATH = Path(
        ALT_ABBREV_OUTPUT_PATH / "alt_abbrev_annotation_manually_annotated_df.xlsx"
    )

## Run LLM with gene symbols, name, and prompt

In [11]:
TEMPERATURES = [0.0]
NUM_RUNS = 3
PROMPT_VERSION = "v1"

sample_name = SAMPLE_PATH.stem
temp_str = "-".join(str(t).replace(".", "p") for t in TEMPERATURES)

experiment_key = build_experiment_key(
    sample_name,
    PROMPT_VERSION,
    TEMPERATURES,
    NUM_RUNS,
)

stored_runs_path = (
    ALT_ABBREV_OUTPUT_PATH / "llm_runs" / f"stored_runs_{experiment_key}.pkl"
)

metadata = {
    "sample_path": str(SAMPLE_PATH),
    "sample_name": sample_name,
    "prompt_version": PROMPT_VERSION,
    "temperatures": TEMPERATURES,
    "num_runs": NUM_RUNS,
}

if stored_runs_path.exists():
    with open(stored_runs_path, "rb") as f:
        saved_data = pickle.load(f)

    if saved_data.get("metadata") == metadata:
        print(f"Loading {stored_runs_path}")
        stored_runs = saved_data["runs"]

    else:
        print(f"Metadata mismatch in {stored_runs_path}. Re-running experiments.")

        stored_runs = run_experiments(df, TEMPERATURES, NUM_RUNS, PROMPT_VERSION)

        with open(stored_runs_path, "wb") as f:
            pickle.dump(
                {
                    "metadata": metadata,
                    "runs": stored_runs,
                },
                f,
            )

else:
    print("Running experiments...")

    stored_runs = run_experiments(df, TEMPERATURES, NUM_RUNS, PROMPT_VERSION)

    with open(stored_runs_path, "wb") as f:
        pickle.dump(
            {
                "metadata": metadata,
                "runs": stored_runs,
            },
            f,
        )

Running experiments...
Running temp=0.0, run=1


T=0.0, run=1:   0%|          | 0/505 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=0.0, run=1
Running temp=0.0, run=2


T=0.0, run=2:   0%|          | 0/505 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=0.0, run=2
Running temp=0.0, run=3


T=0.0, run=3:   0%|          | 0/505 [00:00<?, ?it/s]

prompt_version being stored: v1
Done temp=0.0, run=3


## Summarizes the conditions of the runs stored

In [12]:
sample_names = set()
all_runs = []

for path in (ALT_ABBREV_OUTPUT_PATH / "llm_runs").glob("stored_runs_*.pkl"):
    sample_name = path.stem.replace("stored_runs_", "")
    sample_names.add(sample_name)

    with open(path, "rb") as f:
        data = pickle.load(f)

    for run in data["runs"]:
        run["sample_name"] = sample_name
        all_runs.append(run)

unique_runs = {
    (run["sample_name"], run["prompt_version"], run["temperature"], run["run_idx"])
    for run in all_runs
}

print(f"Files used for runs:{sorted(sample_names)}")
print(f"Number of unique runs: {len(unique_runs)}")
print(f"Unique runs: {unique_runs}")

Files used for runs:['alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3']
Number of unique runs: 6
Unique runs: {('subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 2), ('subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 1), ('alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 1), ('subset_alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 0), ('alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 0), ('alt_abbrev_annotation_manually_annotated_df_pv1_t0p0_agg3', 'v1', 0.0, 2)}


In [13]:
df.write_parquet(stored_runs_path.with_suffix(".parquet"))